In [1]:
import os
import soundfile as sf
import torch
from torch import nn    
import numpy as np
import librosa
from torch.nn import functional as F

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class UNET(nn.Module):
    def __init__(self, in_channels=1, out_channels=2, features=(32, 64, 128, 256)):
        super().__init__()

        self.encoders = nn.ModuleList()
        current = in_channels
        for feature in features:
            self.encoders.append(self.conv_block(current, feature))
            current = feature

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = nn.Sequential(
            self.conv_block(features[-1], features[-1] * 2),
            nn.Dropout2d(0.2)
            )

        self.up_convs = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        
        for feature in reversed(features):
            self.up_convs.append(self.up_conv(feature * 2, feature))
            self.dec_blocks.append(self.conv_block(feature * 2, feature))

        self.final = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=out_channels),
            nn.LeakyReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=out_channels),
            nn.LeakyReLU(inplace=True),
        )

    def up_conv(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(inplace=True),
        )

    def forward(self, x):
        original = x
        skips = []

        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for up, dec, skip in zip(self.up_convs, self.dec_blocks, reversed(skips)):
            x = up(x)           
            x = torch.cat((skip, x), dim=1)
            x = dec(x)

        mask = torch.softmax(self.final(x), dim=1)
        return mask * original

In [4]:
model = UNET(in_channels=1, out_channels=2)
state_dict = torch.load("../models/vocal_inst_sep_unet_best.pt", map_location=device)
model.load_state_dict(state_dict)
model = model.to(device)

In [5]:
SR = 22050  #sampling rate : number of numerical values per second of audio
CHUNK_DURATION = 5
CHUNK_SIZE = SR * CHUNK_DURATION # total number of samples in a chunk
N_FFT = 2048 # number of samples in each FFT window
HOP_LENGTH = 512 # number of samples between successive frames

In [6]:
def load_audio(path):
    audio, sr = librosa.load(path, sr=SR, mono=True)
    audio = np.array(audio)
    return audio

In [7]:
def get_chunks(signal):

    num_chunks = len(signal)//CHUNK_SIZE

    chunks = [signal[i*CHUNK_SIZE:(i+1)*CHUNK_SIZE] for i in range(num_chunks)]
    return chunks

def compute_spectrogram(signal):

    stft = librosa.stft(signal, n_fft=N_FFT, hop_length=HOP_LENGTH)
    magnitude = np.abs(stft)
    phase = np.angle(stft)
    return magnitude,phase

In [8]:

def inference_pipeline(path, model):

    mixture = load_audio(path)
    mixture_chunks = get_chunks(mixture)

    vocals = []
    instruments = []

    model = model.to(device)
    model.eval()

    with torch.no_grad():

        for chunk in mixture_chunks:

            magnitude, phase = compute_spectrogram(chunk)

            spec = torch.tensor(magnitude, dtype=torch.float32)

            # Same padding used during training
            spec = F.pad(spec, (0, 8, 0, 15))

            spec = spec.unsqueeze(0).unsqueeze(0).to(device)

            prediction = model(spec)

            v = prediction[:, 0, :, :].squeeze(0).cpu().numpy()
            i = prediction[:, 1, :, :].squeeze(0).cpu().numpy()

            # Remove training padding
            v = v[:-15, :-8]
            i = i[:-15, :-8]

            vocal_stft = v * np.exp(1j * phase)
            instrument_stft = i * np.exp(1j * phase)

            vocal_audio = librosa.istft(
                vocal_stft,
                n_fft=N_FFT,
                hop_length=HOP_LENGTH,
                length=len(chunk)
            )

            instrument_audio = librosa.istft(
                instrument_stft,
                n_fft=N_FFT,
                hop_length=HOP_LENGTH,
                length=len(chunk)
            )

            vocals.append(vocal_audio)
            instruments.append(instrument_audio)

    vocals = np.concatenate(vocals)
    instruments = np.concatenate(instruments)

    return vocals, instruments


In [9]:
vocals, instruments = inference_pipeline("../test/test_audio.mp3", model)
sf.write("../test/vocals.wav", vocals, SR)
sf.write("../test/instruments.wav", instruments, SR)